In [2]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Input, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
import os

num_skipped = 0
for folder_name in ("Cat", "Dog"):
    folder_path = os.path.join("PetImages", folder_name)
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        try:
            fobj = open(fpath, "rb")
            is_jfif = b"JFIF" in fobj.peek(10)
        finally:
            fobj.close()

        if not is_jfif:
            num_skipped += 1
            # Delete corrupted image
            os.remove(fpath)

print(f"Deleted {num_skipped} images.")

Deleted 0 images.


In [50]:
train_data_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    validation_split=0.2
)

In [51]:
train = train_data_gen.flow_from_directory(
    directory='PetImages',
    target_size=(256,256),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

test = train_data_gen.flow_from_directory(
    directory='PetImages',
    target_size=(256,256),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 18739 images belonging to 2 classes.
Found 4683 images belonging to 2 classes.


In [52]:
model = Sequential()

# Block 1
model.add(Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# Block 2
model.add(Conv2D(64, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# Block 3
model.add(Conv2D(128, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.30))

# Block 4
model.add(Conv2D(256, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.30))

# Classification
model.add(GlobalAveragePooling2D())

model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(1, activation='sigmoid'))

In [53]:
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_12 (Conv2D)          (None, 256, 256, 32)      896       
                                                                 
 batch_normalization_12 (Bat  (None, 256, 256, 32)     128       
 chNormalization)                                                
                                                                 
 max_pooling2d_12 (MaxPoolin  (None, 128, 128, 32)     0         
 g2D)                                                            
                                                                 
 dropout_7 (Dropout)         (None, 128, 128, 32)      0         
                                                                 
 conv2d_13 (Conv2D)          (None, 128, 128, 64)      18496     
                                                                 
 batch_normalization_13 (Bat  (None, 128, 128, 64)    

In [54]:
model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])

In [55]:
model.fit(train, epochs=5, validation_data=test)

Epoch 1/5
586/586 [==============================] - 209s 348ms/step - loss: 0.6794 - accuracy: 0.5917 - val_loss: 0.8368 - val_accuracy: 0.5232
Epoch 2/5
586/586 [==============================] - 203s 345ms/step - loss: 0.6306 - accuracy: 0.6419 - val_loss: 0.6982 - val_accuracy: 0.5924
Epoch 3/5
586/586 [==============================] - 204s 347ms/step - loss: 0.5931 - accuracy: 0.6832 - val_loss: 0.6385 - val_accuracy: 0.6496
Epoch 4/5
586/586 [==============================] - 203s 346ms/step - loss: 0.5478 - accuracy: 0.7178 - val_loss: 0.6785 - val_accuracy: 0.6338
Epoch 5/5
586/586 [==============================] - 205s 349ms/step - loss: 0.4964 - accuracy: 0.7630 - val_loss: 0.4976 - val_accuracy: 0.7640


In [56]:
model.fit(train, initial_epoch=5, epochs=10, validation_data=test)

Epoch 6/10
586/586 [==============================] - 218s 370ms/step - loss: 0.4491 - accuracy: 0.7948 - val_loss: 0.5930 - val_accuracy: 0.7625
Epoch 7/10
586/586 [==============================] - 199s 339ms/step - loss: 0.4076 - accuracy: 0.8134 - val_loss: 0.4756 - val_accuracy: 0.7702
Epoch 8/10
586/586 [==============================] - 197s 336ms/step - loss: 0.3746 - accuracy: 0.8334 - val_loss: 0.3623 - val_accuracy: 0.8341
Epoch 9/10
586/586 [==============================] - 198s 338ms/step - loss: 0.3413 - accuracy: 0.8495 - val_loss: 0.3743 - val_accuracy: 0.8354
Epoch 10/10
586/586 [==============================] - 196s 334ms/step - loss: 0.3172 - accuracy: 0.8647 - val_loss: 0.4214 - val_accuracy: 0.8223


In [59]:
model.fit(train, initial_epoch=10, epochs=15, validation_data=test)

Epoch 11/15
586/586 [==============================] - 200s 341ms/step - loss: 0.2990 - accuracy: 0.8708 - val_loss: 0.4949 - val_accuracy: 0.7813
Epoch 12/15
586/586 [==============================] - 199s 340ms/step - loss: 0.2823 - accuracy: 0.8779 - val_loss: 0.2536 - val_accuracy: 0.8900
Epoch 13/15
586/586 [==============================] - 201s 343ms/step - loss: 0.2740 - accuracy: 0.8839 - val_loss: 0.3046 - val_accuracy: 0.8811
Epoch 14/15
586/586 [==============================] - 198s 338ms/step - loss: 0.2524 - accuracy: 0.8932 - val_loss: 0.2766 - val_accuracy: 0.8834
Epoch 15/15
586/586 [==============================] - 199s 339ms/step - loss: 0.2503 - accuracy: 0.8947 - val_loss: 0.2361 - val_accuracy: 0.9045


In [60]:
model.fit(train, initial_epoch=15, epochs=20, validation_data=test)

Epoch 16/20
586/586 [==============================] - 195s 332ms/step - loss: 0.2417 - accuracy: 0.8982 - val_loss: 0.2438 - val_accuracy: 0.9018
Epoch 17/20
586/586 [==============================] - 198s 337ms/step - loss: 0.2391 - accuracy: 0.9005 - val_loss: 0.6740 - val_accuracy: 0.7997
Epoch 18/20
586/586 [==============================] - 197s 335ms/step - loss: 0.2365 - accuracy: 0.9020 - val_loss: 0.2219 - val_accuracy: 0.9082
Epoch 19/20
586/586 [==============================] - 200s 340ms/step - loss: 0.2310 - accuracy: 0.9059 - val_loss: 0.2240 - val_accuracy: 0.9084
Epoch 20/20
586/586 [==============================] - 200s 341ms/step - loss: 0.2226 - accuracy: 0.9071 - val_loss: 0.2005 - val_accuracy: 0.9178


In [72]:
model.fit(train, initial_epoch=20, epochs=40, validation_data=test)

Epoch 21/40
586/586 [==============================] - 198s 337ms/step - loss: 0.2154 - accuracy: 0.9108 - val_loss: 0.2239 - val_accuracy: 0.9118
Epoch 22/40
586/586 [==============================] - 199s 338ms/step - loss: 0.2175 - accuracy: 0.9076 - val_loss: 0.2695 - val_accuracy: 0.8909
Epoch 23/40
586/586 [==============================] - 200s 341ms/step - loss: 0.2076 - accuracy: 0.9137 - val_loss: 0.1926 - val_accuracy: 0.9171
Epoch 24/40
586/586 [==============================] - 197s 336ms/step - loss: 0.2039 - accuracy: 0.9153 - val_loss: 0.1808 - val_accuracy: 0.9244
Epoch 25/40
586/586 [==============================] - 197s 335ms/step - loss: 0.2038 - accuracy: 0.9153 - val_loss: 0.2074 - val_accuracy: 0.9148
Epoch 26/40
586/586 [==============================] - 199s 339ms/step - loss: 0.1990 - accuracy: 0.9179 - val_loss: 0.1911 - val_accuracy: 0.9242
Epoch 27/40
586/586 [==============================] - 202s 344ms/step - loss: 0.1982 - accuracy: 0.9168 - val_loss: 0

In [75]:
model.fit(train, initial_epoch=40, epochs=60, validation_data=test)

Epoch 41/60
586/586 [==============================] - 197s 335ms/step - loss: 0.1656 - accuracy: 0.9337 - val_loss: 0.2181 - val_accuracy: 0.9105
Epoch 42/60
586/586 [==============================] - 196s 334ms/step - loss: 0.1659 - accuracy: 0.9308 - val_loss: 0.2605 - val_accuracy: 0.8992
Epoch 43/60
586/586 [==============================] - 197s 336ms/step - loss: 0.1676 - accuracy: 0.9316 - val_loss: 0.1942 - val_accuracy: 0.9253
Epoch 44/60
586/586 [==============================] - 199s 339ms/step - loss: 0.1579 - accuracy: 0.9372 - val_loss: 0.1336 - val_accuracy: 0.9483
Epoch 45/60
586/586 [==============================] - 198s 338ms/step - loss: 0.1537 - accuracy: 0.9391 - val_loss: 0.1556 - val_accuracy: 0.9430
Epoch 46/60
586/586 [==============================] - 198s 338ms/step - loss: 0.1568 - accuracy: 0.9380 - val_loss: 0.1573 - val_accuracy: 0.9374
Epoch 47/60
586/586 [==============================] - 196s 335ms/step - loss: 0.1488 - accuracy: 0.9392 - val_loss: 0

In [104]:
model.fit(train, initial_epoch=60, epochs=80, validation_data=test)

Epoch 61/80
586/586 [==============================] - 199s 339ms/step - loss: 0.1449 - accuracy: 0.9415 - val_loss: 0.1451 - val_accuracy: 0.9438
Epoch 62/80
586/586 [==============================] - 197s 336ms/step - loss: 0.1366 - accuracy: 0.9461 - val_loss: 0.1349 - val_accuracy: 0.9453
Epoch 63/80
586/586 [==============================] - 199s 339ms/step - loss: 0.1372 - accuracy: 0.9457 - val_loss: 0.1801 - val_accuracy: 0.9327
Epoch 64/80
586/586 [==============================] - 201s 342ms/step - loss: 0.1368 - accuracy: 0.9447 - val_loss: 0.1936 - val_accuracy: 0.9221
Epoch 65/80
586/586 [==============================] - 201s 342ms/step - loss: 0.1366 - accuracy: 0.9472 - val_loss: 0.1474 - val_accuracy: 0.9483
Epoch 66/80
586/586 [==============================] - 202s 344ms/step - loss: 0.1409 - accuracy: 0.9446 - val_loss: 0.1650 - val_accuracy: 0.9372
Epoch 67/80
586/586 [==============================] - 201s 343ms/step - loss: 0.1350 - accuracy: 0.9456 - val_loss: 0

In [ ]:
import numpy as np

img = tf.keras.utils.load_img(
    "dog.jpg",
    target_size=(256, 256)
)

img_array = tf.keras.utils.img_to_array(img)

img_array = img_array / 255.0 

img_array = np.expand_dims(img_array, axis=0)

In [106]:
pred = model.predict(img_array)[0][0]

print("Raw Prediction:", pred)
print("Classes:", train.class_indices)

if pred > 0.5:
    print("Predicted: Dog")
else:
    print("Predicted: Cat")

1/1 [==============================] - 0s 15ms/step
Raw Prediction: 0.47711185
Classes: {'Cat': 0, 'Dog': 1}
Predicted: Cat
